In [36]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.nn.utils.rnn import pad_sequence
import kagglehub
import os

In [37]:
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

file_path = os.path.join(path, "train.txt")

df = pd.read_csv(file_path, sep=r'\s+', header=None,
                 names=['Word', 'POS', 'Chunk', 'Tag'],
                 skip_blank_lines=False)

Using Colab cache for faster access to the 'conll003-englishversion' dataset.


In [38]:

df["Word"] = df["Word"].astype(str).str.strip() # Ensure 'Word' column is strings and stripped
df["Sentence #"] = (df["Word"] == "-DOCSTART-").cumsum()

df = df[df["Word"] != "-DOCSTART-"]
df = df[df["Word"] != "-"]

sentences = df.groupby("Sentence #").apply(
    lambda s: list(zip(s["Word"], s["Tag"])), include_groups=False
)

sentences = list(sentences)

In [39]:
words = set(df["Word"].values)
tags = set(df["Tag"].values)

word2idx = {w: i+2 for i, w in enumerate(words)}
word2idx["PAD"] = 0
word2idx["UNK"] = 1

tag2idx = {t: i for i, t in enumerate(tags)}

In [40]:
X = [[word2idx.get(w, 1) for w, t in s] for s in sentences]
y = [[tag2idx[t] for w, t in s] for s in sentences]

X = [torch.tensor(seq) for seq in X]
y = [torch.tensor(seq) for seq in y]

X = pad_sequence(X, batch_first=True)
y = pad_sequence(y, batch_first=True)

In [41]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, tag_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.fc = nn.Linear(64, tag_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

model = LSTMModel(len(word2idx), len(tag2idx))

In [42]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [43]:
for epoch in range(3):
    optimizer.zero_grad()

    outputs = model(X)

    outputs = outputs.view(-1, outputs.shape[-1])
    y_flat = y.view(-1)

    loss = criterion(outputs, y_flat)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 2.07122802734375
Epoch 2, Loss: 1.9486308097839355
Epoch 3, Loss: 1.8304790258407593


In [44]:
sample = X[1].unsqueeze(0)
pred = model(sample)

predicted = torch.argmax(pred, dim=-1)[0]

idx2tag = {i: t for t, i in tag2idx.items()}

print("Word | Actual | Predicted")
print("---------------------------")

for i, (word, actual) in enumerate(sentences[1]):
    print(f"{word:10} {actual:8} {idx2tag[predicted[i].item()]}")

Word | Actual | Predicted
---------------------------
nan             nan I-PER
Rare       O        I-PER
Hendrix    B-PER    I-PER
song       O        I-PER
draft      O        I-PER
sells      O        I-ORG
for        O        I-ORG
almost     O        I-ORG
$          O        I-ORG
17,000     O        I-PER
.          O        I-ORG
nan             nan I-ORG
LONDON     B-LOC    I-ORG
1996-08-22 O        I-PER
nan             nan I-PER
A          O        I-ORG
rare       O        I-ORG
early      O        I-ORG
handwritten O        O
draft      O        I-PER
of         O        I-PER
a          O        I-PER
song       O        I-PER
by         O        I-PER
U.S.       B-LOC    I-PER
guitar     O        I-ORG
legend     O        I-ORG
Jimi       B-PER    I-ORG
Hendrix    I-PER    O
was        O        B-ORG
sold       O        I-ORG
for        O        I-ORG
almost     O        I-ORG
$          O        I-ORG
17,000     O        I-PER
on         O        I-PER
Thursday   O     